# Cloud Config Right Sizing Agent

This notebook walks through the full pipeline step by step:

1. **Extract** — Pull text from a SOW PDF
2. **Sanitize** — Redact PII before it reaches the model
3. **Generate** — Call Gemini to produce config JSON (or use Agent Designer)
4. **Validate** — Check the JSON against the CICD schema
5. **Score** — Compare to a known-good config field by field

Each cell can be run independently to see what that step does.

## Setup

Install the required packages. This only needs to run once.

In [ ]:
!pip install -q pymupdf pypdf jsonschema gradio google-genai python-dotenv

---
## Part 1: Extract Text from a PDF

This step takes a PDF file and converts it into plain text.

**Why is this needed?**
AI models work with text, not PDF files. To analyze the content of a SOW,
the text needs to be extracted first. This also enables the sanitizer to
find and redact PII using pattern matching — which can't be done on a raw PDF.

**How it works:**
- Tries PyMuPDF first (better at tables and complex layouts)
- Falls back to pypdf if PyMuPDF returns nothing
- Cleans up messy whitespace from the PDF extraction

In [ ]:
import re

def normalize_whitespace(text):
    """Clean up messy whitespace that PDF extraction often produces."""
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_text_from_pdf(pdf_path):
    """Extract text from a PDF. Tries PyMuPDF first, falls back to pypdf."""
    # Try PyMuPDF (better quality)
    try:
        import fitz
        doc = fitz.open(pdf_path)
        pages = []
        for page_num, page in enumerate(doc, start=1):
            page_text = page.get_text("text").strip()
            if page_text:
                pages.append(f"Page {page_num}\n{page_text}")
        doc.close()
        text = "\n\n".join(pages)
        if text.strip():
            return normalize_whitespace(text)
    except ImportError:
        pass

    # Fallback to pypdf
    try:
        from pypdf import PdfReader
        reader = PdfReader(pdf_path)
        pages = []
        for page_num, page in enumerate(reader.pages, start=1):
            page_text = page.extract_text() or ""
            if page_text.strip():
                pages.append(f"Page {page_num}\n{page_text.strip()}")
        return normalize_whitespace("\n\n".join(pages))
    except ImportError:
        return "Error: Neither PyMuPDF nor pypdf is installed."


print("Extract function ready.")
print("Upload a PDF in the next cell to test it.")

In [ ]:
# Upload a PDF file to test extraction
# In Colab, this opens a file picker dialog
from google.colab import files

print("Select a PDF file to upload...")
uploaded = files.upload()

# Extract text from the first uploaded file
for filename in uploaded:
    print(f"\n=== Extracting text from: {filename} ===")
    raw_text = extract_text_from_pdf(filename)
    print(f"\nExtracted {len(raw_text)} characters.")
    print(f"\n--- First 1000 characters ---\n")
    print(raw_text[:1000])

---
## Part 2: Sanitize — Redact PII

This step takes the extracted text and replaces sensitive customer information
with `[REDACTED]` tags.

**Why is this needed?**
Data privacy is a primary concern. Customer SOWs contain names, emails, phone
numbers, addresses, account IDs, and pricing. None of that should reach the model.

**What gets redacted:** Emails, phones, addresses, account IDs, client names, pricing

**What stays:** Service counts, storage sizes, compute specs, environment type, availability

In [ ]:
# Define the PII patterns
# Each pattern matches a specific type of sensitive data

EMAIL_PATTERN = re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+")
PHONE_PATTERN = re.compile(r"(\+\d{1,2}\s)?\(?\d{3}\)?[\s.-]\d{3}[\s.-]\d{4}")
ADDRESS_PATTERN = re.compile(
    r"\b\d{1,5}\s+[A-Za-z0-9.\- ]+\s(?:Street|St|Road|Rd|Avenue|Ave|Drive|Dr|Lane|Ln|Boulevard|Blvd)\b",
    re.IGNORECASE,
)
ACCOUNT_PATTERN = re.compile(
    r"\b(?:account|contract|customer)[ _-]?(?:id|number|no)\s*[:#-]?\s*[A-Z0-9-]{4,}\b",
    re.IGNORECASE,
)
CLIENT_FIELD_PATTERN = re.compile(
    r"(?im)^(Client Name|Customer Name|Company|Customer|Primary Contact)\s*:\s*.+$"
)
PRICING_PATTERN = re.compile(
    r"(?im)^.*(?:price|pricing|cost|budget|annual spend|monthly spend).*$"
)


def sanitize_text(text):
    """Redact PII from SOW text. Returns sanitized version."""
    if not text:
        return ""
    sanitized = EMAIL_PATTERN.sub("[REDACTED_EMAIL]", text)
    sanitized = PHONE_PATTERN.sub("[REDACTED_PHONE]", sanitized)
    sanitized = ADDRESS_PATTERN.sub("[REDACTED_ADDRESS]", sanitized)
    sanitized = ACCOUNT_PATTERN.sub("[REDACTED_ACCOUNT_ID]", sanitized)
    sanitized = CLIENT_FIELD_PATTERN.sub(
        lambda match: f"{match.group(1)}: [REDACTED]", sanitized
    )
    sanitized = PRICING_PATTERN.sub("[REDACTED_PRICING_LINE]", sanitized)
    sanitized = re.sub(r"\n{3,}", "\n\n", sanitized)
    return sanitized.strip()


print("Sanitizer ready.")

In [ ]:
# Test the sanitizer with a sample
sample_text = """Client Name: Acme Corporation
Contact: john.doe@acme.com
Phone: (555) 123-4567
Address: 123 Main Street, Suite 400

The solution requires 14 microservices with 500 GB of SSD storage.
Environment: production
Budget: $45,000/month
Account ID: ACME-2024-001
High availability is required.
"""

sanitized = sanitize_text(sample_text)

print("=== ORIGINAL ===")
print(sample_text)
print("=== AFTER SANITIZATION ===")
print(sanitized)

In [ ]:
# If a PDF was uploaded in Part 1, sanitize that text
try:
    sanitized_sow = sanitize_text(raw_text)
    print("=== SANITIZED SOW TEXT ===")
    print(sanitized_sow[:2000])
    print(f"\n... ({len(sanitized_sow)} total characters)")
except NameError:
    print("No PDF was uploaded in Part 1. Run those cells first, or skip to Part 3.")

---
## Part 3: Validate JSON Output

This step takes the JSON that the AI (or Agent Designer) generated and checks:
1. Is it valid JSON? (can it be parsed)
2. Does it match the expected schema? (right fields, right types)

**Why is this needed?**
Before any generated config reaches the CICD pipeline, it needs to pass
validation. This acts as a quality gate between the AI and production.

In [ ]:
import json
import jsonschema

# Define the target schema
# On dev day, replace this with the actual CICD schema from Cadence
SCHEMA = {
    "$schema": "http://json-schema.org/draft-07/schema#",
    "type": "object",
    "required": ["customer_name", "environment_tier", "compute", "storage", "networking", "services"],
    "properties": {
        "customer_name": {"type": "string"},
        "environment_tier": {"type": "string", "enum": ["small", "medium", "large", "enterprise"]},
        "compute": {
            "type": "object",
            "required": ["instance_type", "instance_count", "vcpus_per_instance", "memory_gb_per_instance"],
            "properties": {
                "instance_type": {"type": "string"},
                "instance_count": {"type": "integer", "minimum": 1},
                "vcpus_per_instance": {"type": "integer", "minimum": 1},
                "memory_gb_per_instance": {"type": "integer", "minimum": 1}
            }
        },
        "storage": {
            "type": "object",
            "required": ["storage_type", "total_storage_gb", "iops_required"],
            "properties": {
                "storage_type": {"type": "string", "enum": ["pd-standard", "pd-ssd", "pd-balanced"]},
                "total_storage_gb": {"type": "integer", "minimum": 10},
                "iops_required": {"type": "integer", "minimum": 0}
            }
        },
        "networking": {
            "type": "object",
            "required": ["vpc_cidr", "subnet_count", "load_balancer"],
            "properties": {
                "vpc_cidr": {"type": "string"},
                "subnet_count": {"type": "integer", "minimum": 1},
                "load_balancer": {"type": "boolean"}
            }
        },
        "services": {
            "type": "array",
            "items": {
                "type": "object",
                "required": ["service_name", "enabled"],
                "properties": {
                    "service_name": {"type": "string"},
                    "enabled": {"type": "boolean"},
                    "configuration": {"type": "object"}
                }
            }
        },
        "high_availability": {"type": "boolean"},
        "backup_enabled": {"type": "boolean"},
        "estimated_users": {"type": "integer", "minimum": 0}
    }
}


def validate_config(config_json_string, schema):
    """Check if a JSON string is valid and matches the schema."""
    # Step 1: Can it be parsed as JSON?
    try:
        config = json.loads(config_json_string)
    except json.JSONDecodeError as e:
        return False, f"Invalid JSON: {str(e)}"

    # Step 2: Does it match the schema?
    try:
        jsonschema.validate(instance=config, schema=schema)
    except jsonschema.ValidationError as e:
        return False, f"Schema validation failed: {e.message}"

    return True, config


print("Validator ready.")
print(f"Schema has {len(SCHEMA['required'])} required top-level fields.")

In [ ]:
# Paste JSON output from Agent Designer here to validate it
# Replace the example below with real output

agent_output = '''
{
  "customer_name": "Nimbus Retail Solutions",
  "environment_tier": "medium",
  "compute": {
    "instance_type": "n2-standard-4",
    "instance_count": 3,
    "vcpus_per_instance": 4,
    "memory_gb_per_instance": 16
  },
  "storage": {
    "storage_type": "pd-ssd",
    "total_storage_gb": 750,
    "iops_required": 5000
  },
  "networking": {
    "vpc_cidr": "10.5.0.0/16",
    "subnet_count": 3,
    "load_balancer": true
  },
  "services": [
    {"service_name": "Cloud SQL", "enabled": true, "configuration": {}},
    {"service_name": "Cloud Monitoring", "enabled": true, "configuration": {}}
  ],
  "high_availability": true,
  "backup_enabled": true,
  "estimated_users": 500
}
'''

is_valid, result = validate_config(agent_output, SCHEMA)

if is_valid:
    print("PASSED — all required fields present, types correct")
    print(f"\nParsed config:")
    print(json.dumps(result, indent=2))
else:
    print(f"FAILED — {result}")

---
## Part 4: Score Accuracy

This step compares the generated config to a known-good config and reports
how many fields match.

**Why is this needed?**
Passing schema validation means the JSON is structurally correct — but the
values could still be wrong. This step checks: did the AI pick the right
instance type? The right storage size? The right number of services?

In [ ]:
def score_config(generated, expected):
    """Compare two configs field by field. Returns accuracy report."""
    
    # These are the critical fields to check
    # Each tuple: (field_name, parent_key) — parent_key is None for top-level
    # On dev day, update this list based on Cadence's critical fields
    critical_fields = [
        ("environment_tier", None),
        ("instance_type", "compute"),
        ("instance_count", "compute"),
        ("vcpus_per_instance", "compute"),
        ("memory_gb_per_instance", "compute"),
        ("storage_type", "storage"),
        ("total_storage_gb", "storage"),
        ("iops_required", "storage"),
        ("vpc_cidr", "networking"),
        ("subnet_count", "networking"),
        ("load_balancer", "networking"),
        ("high_availability", None),
        ("backup_enabled", None),
        ("estimated_users", None),
    ]

    matched = 0
    total = 0
    details = []

    for field_name, parent_key in critical_fields:
        if parent_key:
            gen_value = generated.get(parent_key, {}).get(field_name)
            exp_value = expected.get(parent_key, {}).get(field_name)
        else:
            gen_value = generated.get(field_name)
            exp_value = expected.get(field_name)

        is_match = gen_value == exp_value
        if is_match:
            matched += 1
        total += 1

        field_label = f"{parent_key}.{field_name}" if parent_key else field_name
        status = "MATCH" if is_match else "MISMATCH"
        details.append((field_label, exp_value, gen_value, status))

    # Check services
    gen_services = sorted([s["service_name"] for s in generated.get("services", [])])
    exp_services = sorted([s["service_name"] for s in expected.get("services", [])])
    services_match = gen_services == exp_services
    if services_match:
        matched += 1
    total += 1
    details.append(("services (names)", exp_services, gen_services, "MATCH" if services_match else "MISMATCH"))

    accuracy = (matched / total * 100) if total > 0 else 0
    return round(accuracy, 1), matched, total, details


print("Scorer ready.")

In [ ]:
# Define the known-good config (the correct answer)
# On dev day, replace this with the real expected config from Cadence

known_good = {
    "customer_name": "Nimbus Retail Solutions",
    "environment_tier": "medium",
    "compute": {
        "instance_type": "n2-standard-4",
        "instance_count": 3,
        "vcpus_per_instance": 4,
        "memory_gb_per_instance": 16
    },
    "storage": {
        "storage_type": "pd-ssd",
        "total_storage_gb": 750,
        "iops_required": 5000
    },
    "networking": {
        "vpc_cidr": "10.5.0.0/16",
        "subnet_count": 3,
        "load_balancer": True
    },
    "services": [
        {"service_name": "Cloud SQL", "enabled": True, "configuration": {"engine": "PostgreSQL"}},
        {"service_name": "Cloud Storage", "enabled": True, "configuration": {}},
        {"service_name": "Cloud CDN", "enabled": True, "configuration": {}},
        {"service_name": "Cloud Monitoring", "enabled": True, "configuration": {}},
        {"service_name": "Cloud Logging", "enabled": True, "configuration": {}}
    ],
    "high_availability": True,
    "backup_enabled": True,
    "estimated_users": 500
}

# Score the agent output against the known-good config
generated = json.loads(agent_output)
accuracy, matched, total, details = score_config(generated, known_good)

print(f"ACCURACY: {accuracy}% ({matched}/{total} fields)")
print()
print(f"{'Field':<35} {'Expected':<20} {'Generated':<20} {'Result'}")
print("-" * 85)
for field, expected, got, status in details:
    print(f"{field:<35} {str(expected)[:18]:<20} {str(got)[:18]:<20} {status}")

---
## Part 5: Launch the Full Gradio App (Optional)

This launches the complete 3-tab app with a shareable link.
Anyone on the team can open the link in their browser.

In [ ]:
# Clone the repo and launch the full app
!git clone https://github.com/mikaelaconnell-datapiper/cadence-cloud-config.git
%cd cadence-cloud-config
!pip install -q -r requirements.txt

In [ ]:
# Launch the Gradio app with a public share link
!python app.py